<a href="https://colab.research.google.com/github/impos0108/AI4WeatherandClimate/blob/main/graphcast_1deg_infer_philippines_with_exp_ENG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌏 Hands-On AI Weather Forecasting
## Predicting Philippines Precipitation with GraphCast

---

### What You Will Learn

In this notebook, you will run **GraphCast**, an AI weather forecasting model developed by Google DeepMind.

Traditional numerical weather prediction (NWP) takes supercomputers **hours** to compute.  
GraphCast completes the same forecast in **minutes**. How is that possible?

---

### 🤔 How Does AI Weather Forecasting Work?

Traditional NWP solves physical equations (fluid dynamics, thermodynamics, etc.) step by step.  
AI weather models instead learn patterns from **40 years of historical weather data (ERA5)**:

> *"When the atmosphere looked like this, here is what happened 6 hours later."*

```
Input:  Current atmospheric state
        (temperature, pressure, wind, humidity × 13 vertical levels)
            ↓  [GraphCast neural network]
Output: Predicted state 6 hours later
            ↓  Repeat (autoregressive)
Output: +12h → +18h → ... → +66h
```

---

### 📋 Notebook Overview

| Item | Details |
|------|---------|
| Model | GraphCast_small (1° resolution, 13 pressure levels) |
| Input data | WeatherBench2 public ERA5 (no authentication required) |
| Initial time | **2021-08-05 00 UTC** (Philippines typhoon season) |
| Forecast range | +6h to +66h (6-hour steps, 11 total) |
| Validation | Compare AI forecast vs. ERA5 reanalysis |

> ⚠️ **GPU required**: Runtime → Change runtime type → Select **T4 GPU** before running

---


## 🔧 Step 1. Install Required Libraries

> **What is a library?** A library is a collection of pre-written code that you can reuse.  
> Instead of writing everything from scratch, we install libraries with `pip install`.

Key libraries used in this notebook:

| Library | Purpose |
|---------|---------|
| **jax** | Google's high-performance numerical computing (NumPy + GPU acceleration) |
| **haiku** | JAX-based neural network framework (used by GraphCast) |
| **xarray** | Handles multi-dimensional scientific data (NetCDF, Zarr) |
| **cartopy** | Draws meteorological data on maps |

⏱️ Installation takes **2–3 minutes**. Click the ▶ button on the left to run each cell.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "--upgrade", "jax[cuda12]",
    "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"],
    check=True)

pkgs = [
    "dm-haiku", "jaxlib", "chex", "jraph", "trimesh",
    "toolz>=0.11,<1",
    "xarray", "scipy", "netCDF4", "h5netcdf",
    "matplotlib", "pandas", "cartopy",
    "google-cloud-storage", "huggingface_hub",
    "cdsapi",
]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkg.split(),
                   check=True)
print("Package installation complete")


## 📦 Step 2. Install GraphCast from GitHub

GraphCast is **open-source** — DeepMind released the code and model weights publicly.  
We clone the repository directly from GitHub and install it as a Python package.

> **GitHub** is a platform where developers share code publicly.  
> Anyone can view, download, and use open-source projects for free.

- 📄 Paper: [Lam et al. (2023), Science](https://www.science.org/doi/10.1126/science.adi2336)
- 💻 GitHub: https://github.com/google-deepmind/graphcast


In [ ]:
import subprocess, sys, os

REPO_DIR = "/content/graphcast_repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/google-deepmind/graphcast.git", REPO_DIR],
                   check=True)
    print("Clone complete")
else:
    print("Already cloned, skipping")

req = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jraph"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from graphcast import graphcast, checkpoint, normalization, autoregressive, casting
print("GraphCast import successful")


## 🔑 Step 3. Google Account Authentication

We need to authenticate with Google to download the model checkpoint  
from Google Cloud Storage. Use the same Google account you used to open Colab.

Running the cell below will open an authentication popup.


In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Google authentication complete")


## 📅 Step 4. Set Forecast Date and File Paths

Here we set the **initial time** — the starting point for the AI forecast.

### Why use a past date?
GraphCast needs **ERA5 reanalysis data** as input.  
ERA5 is a historical weather database; the WeatherBench2 public dataset covers **1959–2023**.

Using a past date means we can also **validate** the forecast by comparing it  
against what actually happened (the ERA5 "truth" data) — which we do in Step 19!

### How to change the date
Edit `INIT_DT = datetime(2021, 8, 5, 0)` to any date within **1959-01-01 to 2023-01-09**.

Some interesting dates to try:
```python
INIT_DT = datetime(2013, 11, 8, 0)   # Super Typhoon Haiyan — just before landfall
INIT_DT = datetime(2020, 10, 31, 0)  # Typhoon Goni — strongest landfalling typhoon on record
INIT_DT = datetime(2021,  8,  5, 0)  # Current setting: active typhoon season
```


In [ ]:
from datetime import datetime, timedelta
import os

# Use a historical date within WeatherBench2 ERA5 range (1959–2023)
# 2021-08-05: active typhoon season over the Philippines
INIT_DT        = datetime(2021, 8, 5, 0)
FORECAST_START = datetime(2021, 8, 5, 6)
FORECAST_END   = datetime(2021, 8, 7, 18)
N_STEPS = int((FORECAST_END - FORECAST_START).total_seconds() / 3600 / 6) + 1

print(f"Initial time  : {INIT_DT} UTC")
print(f"Forecast steps: {N_STEPS}  ({N_STEPS * 6}h)")

PARAM_DIR   = "/content/graphcast_params"
STATS_DIR   = "/content/graphcast_stats"
DATA_DIR    = "/content/graphcast_data"
OUTPUT_DIR  = "/content/graphcast_output"
for d in [PARAM_DIR, STATS_DIR, DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# GraphCast_small checkpoint (1° / 13 levels / mesh 2to5)
CKPT_NAME   = ("GraphCast_small - ERA5 1979-2015 - resolution 1.0 "
               "- pressure levels 13 - mesh 2to5 - precipitation input and output.npz")
CKPT_PATH   = f"{PARAM_DIR}/{CKPT_NAME}"
STATS_PATH  = f"{STATS_DIR}/diffs_stddev_by_level.nc"
MEAN_PATH   = f"{STATS_DIR}/mean_by_level.nc"
STDDEV_PATH = f"{STATS_DIR}/stddev_by_level.nc"
ERA5_PATH   = f"{DATA_DIR}/era5_wb2_1deg_{INIT_DT.strftime('%Y%m%d%H')}.nc"
TAG         = INIT_DT.strftime("%Y%m%d%H")

# WeatherBench2 public Zarr path (anonymous, no auth)
WB2_ZARR = ("gs://weatherbench2/datasets/era5/"
            "1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr")

print(f"\nCheckpoint : {CKPT_NAME}")
print(f"WB2 Zarr   : {WB2_ZARR}")


## 🧠 Step 5. Download Model Checkpoint (Weights)

### What is a model checkpoint?
A deep learning model consists of hundreds of millions of numbers called **parameters** (or weights).  
After training, these numbers are saved to a file called a **checkpoint**.  
We simply download DeepMind's pre-trained checkpoint — no training required!

### GraphCast_small Specifications

| Item | Value |
|------|-------|
| Training data | ERA5 1979–2015 (37 years) |
| Spatial resolution | 1° (~111 km) |
| Vertical levels | 13 pressure levels (50–1000 hPa) |
| File size | ~340 MB |
| GPU memory needed | ~12–14 GB (fits on T4) |

> 💡 A higher-resolution 0.25° model also exists, but requires an A100 80 GB GPU.  
> In this workshop we use `GraphCast_small`, which runs on a free Colab T4.

⏱️ First-time download takes **1–2 minutes**.


In [ ]:
import os, shutil
from google.cloud import storage as gcs

BUCKET = "dm_graphcast"

def download_from_gcs_stream(blob_path, local_path):
    client = gcs.Client.create_anonymous_client()
    bucket = client.bucket(BUCKET)
    for prefix in ["graphcast/", ""]:
        full_path = f"{prefix}{blob_path}"
        blob = bucket.blob(full_path)
        try:
            if blob.exists():
                print(f"  GCS -> {os.path.basename(local_path)}")
                blob.download_to_filename(local_path)
                print(f"  Done ({os.path.getsize(local_path)/1e6:.1f} MB)")
                return True
        except Exception as e:
            print(f"  GCS '{full_path}' failed: {e}")
    return False

def download_from_hf(repo_id, filename, local_path, repo_type="model"):
    from huggingface_hub import hf_hub_download
    print(f"  HuggingFace: {repo_id} / {filename}")
    src = hf_hub_download(repo_id=repo_id, filename=filename, repo_type=repo_type)
    shutil.copy(src, local_path)
    print(f"  Done ({os.path.getsize(local_path)/1e6:.1f} MB)")

# Checkpoint
print("Downloading GraphCast_small checkpoint (~340 MB)...")
if not os.path.exists(CKPT_PATH):
    # 1) Official GCS bucket
    ok = download_from_gcs_stream(f"params/{CKPT_NAME}", CKPT_PATH)

    if not ok:
        # 2) HuggingFace mirror (correct repo for the small model)
        try:
            download_from_hf("shermansiu/dm_graphcast_small", CKPT_NAME, CKPT_PATH)
            ok = True
        except Exception as e:
            print(f"  HF shermansiu/dm_graphcast_small failed: {e}")

    if not ok:
        cmd = f"gsutil cp gs://dm_graphcast/graphcast/params/'{CKPT_NAME}' '{CKPT_PATH}'"
        raise RuntimeError(
            "All download sources failed.\n"
            "Try downloading manually with gsutil:\n"
            f"  {cmd}"
        )
else:
    print(f"  Already exists: {os.path.basename(CKPT_PATH)}")

# Normalization statistics
print("\nDownloading normalization statistics...")
for fname in ["diffs_stddev_by_level.nc", "mean_by_level.nc", "stddev_by_level.nc"]:
    local = f"{STATS_DIR}/{fname}"
    if not os.path.exists(local):
        ok = download_from_gcs_stream(f"stats/{fname}", local)
        if not ok:
            try:
                download_from_hf("shermansiu/dm_graphcast_datasets",
                                 f"stats/{fname}", local, repo_type="dataset")
            except Exception as e:
                raise RuntimeError(f"Failed to download {fname}: {e}")
    else:
        print(f"  Already exists: {fname}")

print("\nAll model files ready")


## 🌐 Step 6. (Skipped) CDS API Setup

> ✅ **No CDS API key needed for this workshop!**  
> We use WeatherBench2 public data, which requires no account or API key.  
> You can skip the cell below entirely.

*Note: If you want to run forecasts for dates after January 2023,  
you would need to download ERA5 directly from Copernicus CDS.*


In [ ]:
# ✅ You can skip this cell (not needed when using WeatherBench2 data).
# Run this only if you have a CDS API key.

# import os, getpass
# CDS_KEY = getpass.getpass("Personal Access Token: ").strip()
# cds_rc = os.path.expanduser("~/.cdsapirc")
# with open(cds_rc, "w") as f:
#     f.write("url: https://cds.climate.copernicus.eu/api\n")
#     f.write(f"key: {CDS_KEY}\n")
# print("CDS API configuration complete")

print("Step 6 skipped — using WeatherBench2 public data instead.")


## 📡 Step 7. Load ERA5 Input Data from WeatherBench2 (No Auth Required)

### What is ERA5?
**ERA5** is a global atmospheric reanalysis dataset produced by ECMWF (European Centre for Medium-Range Weather Forecasts).  
It combines millions of observations (satellites, radiosondes, surface stations) with a numerical model  
to reconstruct past weather at 0.25° resolution. GraphCast was trained on 40 years of this data.

### WeatherBench2 Public Bucket
Google has made ERA5 available in **Zarr format** on a public GCS bucket — no account needed:
```
gs://weatherbench2/datasets/era5/   ← public, anonymous access
```

### What GraphCast Needs as Input
GraphCast takes **two consecutive time steps** as input:
- `t − 6h`: 6 hours before the initial time
- `t0`: the initial time (forecast start)

The difference between the two time steps tells the model the "direction of motion" of the atmosphere.

⏱️ Zarr streams only the 2 required time steps — no need to download the full dataset (~30 sec to 2 min).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gcsfs", "zarr"], check=True)

import xarray as xr
import numpy as np
import os

def load_era5_from_wb2(zarr_path, init_dt, era5_path):
    if os.path.exists(era5_path):
        print(f"Cache exists, loading: {era5_path}")
        return xr.open_dataset(era5_path)

    print(f"Opening WB2 Zarr (anonymous)...")
    ds = xr.open_zarr(zarr_path, storage_options={"token": "anon"}, consolidated=True)

    print(f"  Available variables ({len(ds.data_vars)}): {sorted(ds.data_vars)}")

    # Select two time steps: t-6h and t0
    t0 = np.datetime64(init_dt)
    t1 = np.datetime64(init_dt - timedelta(hours=6))
    print(f"  Selecting: {[str(t)[:16] for t in [t1, t0]]}")
    ds_sel = ds.sel(time=[t1, t0]).compute()  # pull into memory

    # Rename lat/lon dims
    rename_map = {}
    if "latitude" in ds_sel.dims:  rename_map["latitude"] = "lat"
    if "longitude" in ds_sel.dims: rename_map["longitude"] = "lon"
    if rename_map:
        ds_sel = ds_sel.rename(rename_map)

    # Rename total_precipitation -> total_precipitation_6hr
    if "total_precipitation" in ds_sel and "total_precipitation_6hr" not in ds_sel:
        ds_sel = ds_sel.rename({"total_precipitation": "total_precipitation_6hr"})

    # Regrid 0.25° -> 1° by coarsening
    print("  Regridding 0.25° -> 1° ...")
    ds_1deg = ds_sel.coarsen(lat=4, lon=4, boundary="trim").mean()

    lat = ds_1deg.lat.values
    lon = ds_1deg.lon.values
    times = ds_1deg.time.values

    # Helper: make a (time, lat, lon) zero array
    def zeros_sfc():
        return xr.DataArray(np.zeros((2, len(lat), len(lon)), dtype=np.float32),
                            dims=["time","lat","lon"],
                            coords={"time": times, "lat": lat, "lon": lon})

    # ── Derive / fill missing variables ───────────────────────────────────────
    # geopotential_at_surface — use lowest pressure-level geopotential at t0
    if "geopotential_at_surface" not in ds_1deg:
        if "geopotential" in ds_1deg:
            ds_1deg["geopotential_at_surface"] = (
                ds_1deg["geopotential"]
                .isel(level=-1, time=0)
                .drop_vars(["level","time"], errors="ignore")
            )
            print("  Derived geopotential_at_surface")
        else:
            ds_1deg["geopotential_at_surface"] = xr.DataArray(
                np.zeros((len(lat), len(lon)), dtype=np.float32),
                dims=["lat","lon"], coords={"lat": lat, "lon": lon})
            print("  geopotential_at_surface set to 0 (not in WB2)")

    # land_sea_mask — fill with 0 if missing
    if "land_sea_mask" not in ds_1deg:
        ds_1deg["land_sea_mask"] = xr.DataArray(
            np.zeros((len(lat), len(lon)), dtype=np.float32),
            dims=["lat","lon"], coords={"lat": lat, "lon": lon})
        print("  land_sea_mask set to 0 (not in WB2)")

    # sea_surface_temperature — use 2m_temperature as proxy if missing
    if "sea_surface_temperature" not in ds_1deg:
        if "2m_temperature" in ds_1deg:
            ds_1deg["sea_surface_temperature"] = ds_1deg["2m_temperature"].copy()
            print("  sea_surface_temperature proxied from 2m_temperature")
        else:
            ds_1deg["sea_surface_temperature"] = zeros_sfc()

    # sea_ice_cover
    if "sea_ice_cover" not in ds_1deg:
        ds_1deg["sea_ice_cover"] = zeros_sfc()
        print("  sea_ice_cover set to 0 (not in WB2)")

    # vertical_velocity (omega) — fill with 0 if missing
    if "vertical_velocity" not in ds_1deg:
        if "level" in ds_1deg.dims:
            levels = ds_1deg.level.values
            ds_1deg["vertical_velocity"] = xr.DataArray(
                np.zeros((2, len(levels), len(lat), len(lon)), dtype=np.float32),
                dims=["time","level","lat","lon"],
                coords={"time": times, "level": levels, "lat": lat, "lon": lon})
        print("  vertical_velocity set to 0 (not in WB2)")

    # toa_incident_solar_radiation — fill with 0 if missing
    if "toa_incident_solar_radiation" not in ds_1deg:
        ds_1deg["toa_incident_solar_radiation"] = zeros_sfc()
        print("  toa_incident_solar_radiation set to 0 (not in WB2)")

    # total_precipitation_6hr — fill with 0 if missing
    if "total_precipitation_6hr" not in ds_1deg:
        ds_1deg["total_precipitation_6hr"] = zeros_sfc()
        print("  total_precipitation_6hr set to 0 (not in WB2)")

    # Fill NaN in SST / sea ice
    for v in ["sea_surface_temperature","sea_ice_cover"]:
        ds_1deg[v] = ds_1deg[v].fillna(0.0)

    print(f"  Saving cache: {era5_path}")
    ds_1deg.to_netcdf(era5_path)
    print(f"  Done ({os.path.getsize(era5_path)/1e6:.1f} MB)")
    return ds_1deg

era5_ds = load_era5_from_wb2(WB2_ZARR, INIT_DT, ERA5_PATH)
print("\nERA5 loaded successfully")
print(f"  lat: {len(era5_ds.lat)} pts  ({float(era5_ds.lat[0]):.1f} ~ {float(era5_ds.lat[-1]):.1f})")
print(f"  lon: {len(era5_ds.lon)} pts  ({float(era5_ds.lon[0]):.1f} ~ {float(era5_ds.lon[-1]):.1f})")
if "level" in era5_ds.dims:
    print(f"  levels: {list(era5_ds.level.values)}")
print(f"  variables ({len(era5_ds.data_vars)}): {sorted(era5_ds.data_vars)}")


## 🔬 Step 8. Import Libraries and Verify GPU

Now we import all the libraries we installed and check that JAX can see the **T4 GPU**.

> **JAX** is Google's scientific computing library.  
> It works like NumPy but automatically uses the GPU for fast parallel computation.  
> All of GraphCast's math runs through JAX.

⚠️ If no GPU appears: Runtime → Change runtime type → T4 GPU → **restart and run all cells again**.


In [ ]:
import sys, os, logging, dataclasses
from datetime import datetime, timedelta

REPO_DIR = "/content/graphcast_repo"
if os.path.exists(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import xarray as xr
import pandas as pd
import jax
import jax.numpy as jnp
import haiku as hk
import functools

from graphcast import (graphcast, checkpoint, normalization,
                       autoregressive, casting, model_utils)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
logger = logging.getLogger("graphcast")

print(f"JAX version : {jax.__version__}")
print(f"JAX devices : {jax.devices()}")

# Check T4 GPU memory
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader"], capture_output=True, text=True)
print(f"GPU         : {result.stdout.strip()}")


## 🏗️ Step 9. Load Model Architecture and Normalization Statistics

### GraphCast Architecture
GraphCast is built on a **Graph Neural Network (GNN)**:
```
Encoder  (grid → icosahedral mesh)
    ↓
Processor  (12 GNN layers — nodes exchange information with neighbors)
    ↓
Decoder  (icosahedral mesh → grid)
```
The Earth is divided into an icosahedral mesh where each node communicates  
with its neighbors to propagate atmospheric information across the globe.

### Why Normalization Statistics?
Neural networks are sensitive to the scale of inputs.  
For example, temperature (~250 K) and surface pressure (~100,000 Pa) have very different magnitudes.  
We load statistics computed during training to **standardize** each variable the same way:

| File | Contents |
|------|---------|
| `mean_by_level.nc` | Per-variable mean values |
| `stddev_by_level.nc` | Per-variable standard deviations |
| `diffs_stddev_by_level.nc` | Standard deviation of 6-hour differences |


In [ ]:
def load_checkpoint(path):
    logger.info(f"Loading checkpoint: {os.path.basename(path)}")
    with open(path, "rb") as f:
        ckpt = checkpoint.load(f, graphcast.CheckPoint)
    logger.info(f"  Resolution: {ckpt.model_config.resolution}° | "
                f"Pressure levels: {len(ckpt.task_config.pressure_levels)}")
    has_precip = any("precip" in v.lower() for v in ckpt.task_config.target_variables)
    logger.info(f"  Precipitation variable included: {has_precip}")
    return ckpt.params, {}, ckpt.model_config, ckpt.task_config

params, state, model_config, task_config = load_checkpoint(CKPT_PATH)
diffs_stddev = xr.load_dataset(STATS_PATH)
mean         = xr.load_dataset(MEAN_PATH)
stddev       = xr.load_dataset(STDDEV_PATH)
print("\nModel ready")
print(f"Pressure levels: {list(task_config.pressure_levels)}")


## 🔄 Step 10. Preprocess ERA5 Input Data

We convert ERA5 data into the exact format GraphCast expects.

### Variable Name Mapping
ERA5 uses short names; GraphCast expects full descriptive names:

| ERA5 short name | GraphCast variable name | Description |
|----------------|------------------------|-------------|
| z | geopotential | Geopotential height |
| t | temperature | Air temperature |
| u | u_component_of_wind | East–west wind |
| v | v_component_of_wind | North–south wind |
| q | specific_humidity | Water vapor content |
| w | vertical_velocity | Vertical air motion |

### Constant Variables (time-invariant)
- `land_sea_mask`: 0 = ocean, 1 = land
- `geopotential_at_surface`: surface elevation encoded as geopotential (contains mountain/terrain info)


In [ ]:
ERA5_RENAME = {
    "z":      "geopotential",
    "t":      "temperature",
    "u":      "u_component_of_wind",
    "v":      "v_component_of_wind",
    "q":      "specific_humidity",
    "w":      "vertical_velocity",
    "t2m":    "2m_temperature",
    "msl":    "mean_sea_level_pressure",
    "u10":    "10m_u_component_of_wind",
    "v10":    "10m_v_component_of_wind",
    "tcwv":   "total_column_water_vapour",
    "sst":    "sea_surface_temperature",
    "siconc": "sea_ice_cover",
    "tisr":   "toa_incident_solar_radiation",
    "tp":     "total_precipitation_6hr",
    "lsm":    "land_sea_mask",
    "level":       "level",
    "pressure_level": "level",
    "latitude":    "lat",
    "longitude":   "lon",
}

FORCING_ONLY_VARS = [
    "toa_incident_solar_radiation",
    "year_progress_sin", "year_progress_cos",
    "day_progress_sin",  "day_progress_cos",
]
CONSTANT_VARS = ["geopotential_at_surface", "land_sea_mask"]

def load_era5(path, task_config, init_dt):
    logger.info(f"Loading ERA5: {path}")
    ds = xr.open_dataset(path)

    rename_map = {k: v for k, v in ERA5_RENAME.items()
                  if k in list(ds.data_vars) + list(ds.dims)}
    ds = ds.rename(rename_map)

    if "pressure_level" in ds.dims and "level" not in ds.dims:
        ds = ds.rename({"pressure_level": "level"})

    # Create geopotential_at_surface (lat, lon constant)
    if "geopotential_at_surface" not in ds and "geopotential" in ds:
        ds["geopotential_at_surface"] = (
            ds["geopotential"].isel(level=-1, time=0)
            .drop_vars(["level", "time"])
        )

    # Keep only (lat, lon) for constant variables
    for cvar in CONSTANT_VARS:
        if cvar in ds:
            drop_dims = [c for c in ["time", "level", "batch"] if c in ds[cvar].dims]
            if drop_dims:
                ds[cvar] = ds[cvar].isel(**{c: 0 for c in drop_dims}).drop_vars(drop_dims)

    # Fill missing values
    for var in ["sea_surface_temperature", "sea_ice_cover"]:
        if var in ds:
            ds[var] = ds[var].fillna(0.0)

    # Select two time steps
    t0 = np.datetime64(init_dt)
    t1 = np.datetime64(init_dt - timedelta(hours=6))
    ds = ds.sel(time=[t1, t0])

    required  = list(task_config.input_variables)
    available = [v for v in required if v in ds]
    missing   = [v for v in required if v not in ds]
    if missing:
        logger.warning(f"  Missing variables ({len(missing)}): {missing[:4]} ...")

    logger.info(f"  Variables available: {len(available)} / {len(required)} required")
    return ds[available]

era5_ds = load_era5(ERA5_PATH, task_config, INIT_DT)
print("ERA5 loaded successfully")
print(f"  Grid size: lat={len(era5_ds.lat)}, lon={len(era5_ds.lon)}")
print(f"  Pressure levels: {len(era5_ds.level)}")


## ⚡ Step 11. JIT Compilation (Just-In-Time Compilation)

### What is JIT?
Python is generally a slow language for heavy computation.  
JAX's `@jax.jit` decorator **compiles** the Python function into optimized GPU machine code  
the first time it is called:

```
First call:  compile Python → GPU machine code  (takes 1–3 min) 😴
Later calls: run compiled code directly          (very fast 🚀)
```

> 💡 Think of it as converting a Python script into C++ speed automatically.

⏱️ This step takes **1–3 minutes**. While waiting, let's understand how GraphCast makes predictions:

### Autoregressive Forecasting
```
[t−6h, t0]       → model → predict t+6h
[t0,   t+6h]     → model → predict t+12h
[t+6h, t+12h]    → model → predict t+18h
... repeat 11 times ...
→ final: t+66h forecast
```
Each output becomes the next input. This is called **autoregressive** prediction.


In [ ]:
def build_jit_runner(params, state, model_config, task_config,
                     diffs_stddev, mean, stddev):
    logger.info("Compiling JIT... (1° model typically takes ~1–2 minutes on T4)")

    def construct_wrapped_graphcast():
        predictor = graphcast.GraphCast(model_config, task_config)
        predictor = casting.Bfloat16Cast(predictor)
        predictor = normalization.InputsAndResiduals(
            predictor,
            diffs_stddev_by_level=diffs_stddev,
            mean_by_level=mean,
            stddev_by_level=stddev,
        )
        predictor = autoregressive.Predictor(predictor, gradient_checkpointing=False)
        return predictor

    @hk.transform_with_state
    def run_forward(inputs, targets_template, forcings):
        return construct_wrapped_graphcast()(
            inputs, targets_template, forcings, is_training=False)

    @jax.jit
    def run_forward_jit(inputs, targets_template, forcings):
        return run_forward.apply(
            params, state, jax.random.PRNGKey(0),
            inputs, targets_template, forcings)

    logger.info("JIT compilation complete")
    return run_forward_jit

jit_fn = build_jit_runner(
    params, state, model_config, task_config,
    diffs_stddev, mean, stddev,
)


## 🚀 Step 12. Prepare Inputs and Run Inference

Time to run the AI model!

### What are Forcing Variables?
Some inputs are not predicted by the model — they are **externally prescribed** at each forecast step:

| Variable | Description |
|----------|-------------|
| `toa_incident_solar_radiation` | Solar radiation at top of atmosphere (computed from geometry) |
| `year_progress_sin/cos` | Season information encoded as sine/cosine |
| `day_progress_sin/cos` | Time-of-day information encoded as sine/cosine |

These are computed from physics and are known exactly for any future time — no prediction needed.

### Expected Runtime
Running 11 forecast steps (+6h to +66h) on a T4 GPU takes approximately **2–5 minutes**.  
A traditional numerical weather model would take hours on a supercomputer!


In [ ]:
def make_forcings(era5_ds, target_times):
    lat = era5_ds.lat.values
    lon = era5_ds.lon.values
    n_times = len(target_times)

    yp_sin, yp_cos, dp_sin, dp_cos = [], [], [], []
    for t in target_times:
        ts = pd.Timestamp(t.astype("datetime64[ms]").astype(datetime))
        yp = (ts.timetuple().tm_yday - 1) / 365.0
        dp = (ts.hour / 24.0 + lon / 360.0) % 1.0
        yp_sin.append(np.sin(2*np.pi*yp))
        yp_cos.append(np.cos(2*np.pi*yp))
        dp_sin.append(np.sin(2*np.pi*dp))
        dp_cos.append(np.cos(2*np.pi*dp))

    if "toa_incident_solar_radiation" in era5_ds:
        tisr_base = era5_ds["toa_incident_solar_radiation"].isel(time=-1).values
        tisr = np.stack([tisr_base]*n_times, axis=0)
    else:
        tisr = np.zeros((n_times, len(lat), len(lon)))

    return xr.Dataset({
        "toa_incident_solar_radiation": xr.DataArray(
            tisr[np.newaxis], dims=["batch","time","lat","lon"],
            coords={"lat": lat, "lon": lon, "time": target_times}),
        "year_progress_sin": xr.DataArray(
            np.array(yp_sin)[np.newaxis, :], dims=["batch","time"],
            coords={"time": target_times}),
        "year_progress_cos": xr.DataArray(
            np.array(yp_cos)[np.newaxis, :], dims=["batch","time"],
            coords={"time": target_times}),
        "day_progress_sin": xr.DataArray(
            np.array(dp_sin)[np.newaxis], dims=["batch","time","lon"],
            coords={"lon": lon, "time": target_times}),
        "day_progress_cos": xr.DataArray(
            np.array(dp_cos)[np.newaxis], dims=["batch","time","lon"],
            coords={"lon": lon, "time": target_times}),
    })

def make_inputs_forcings_for_inputs(era5_ds, input_times):
    lat = era5_ds.lat.values
    lon = era5_ds.lon.values
    yp_sin, yp_cos, dp_sin, dp_cos = [], [], [], []
    for t in input_times:
        ts = pd.Timestamp(t.astype("datetime64[ms]").astype(datetime))
        yp = (ts.timetuple().tm_yday - 1) / 365.0
        dp = (ts.hour / 24.0 + lon / 360.0) % 1.0
        yp_sin.append(np.sin(2*np.pi*yp))
        yp_cos.append(np.cos(2*np.pi*yp))
        dp_sin.append(np.sin(2*np.pi*dp))
        dp_cos.append(np.cos(2*np.pi*dp))

    if "toa_incident_solar_radiation" in era5_ds:
        tisr = era5_ds["toa_incident_solar_radiation"].values
    else:
        tisr = np.zeros((len(input_times), len(lat), len(lon)))

    return xr.Dataset({
        "toa_incident_solar_radiation": xr.DataArray(
            tisr, dims=["time","lat","lon"],
            coords={"lat": lat, "lon": lon, "time": input_times}),
        "year_progress_sin": xr.DataArray(
            np.array(yp_sin), dims=["time"], coords={"time": input_times}),
        "year_progress_cos": xr.DataArray(
            np.array(yp_cos), dims=["time"], coords={"time": input_times}),
        "day_progress_sin": xr.DataArray(
            np.array(dp_sin), dims=["time","lon"],
            coords={"lon": lon, "time": input_times}),
        "day_progress_cos": xr.DataArray(
            np.array(dp_cos), dims=["time","lon"],
            coords={"lon": lon, "time": input_times}),
    })

def prepare_inputs_and_targets(ds, task_config, init_dt, n_steps):
    target_times = np.array([
        np.datetime64(init_dt + timedelta(hours=(i+1)*6))
        for i in range(n_steps)
    ])
    input_times = ds.time.values

    input_forcing_ds = make_inputs_forcings_for_inputs(ds, input_times)
    inputs_ds = xr.merge([
        ds.drop_vars(CONSTANT_VARS, errors="ignore"),
        input_forcing_ds,
    ])
    for cvar in CONSTANT_VARS:
        if cvar in ds:
            inputs_ds[cvar] = ds[cvar]

    avail  = [v for v in task_config.input_variables if v in inputs_ds]
    inputs = inputs_ds[avail].expand_dims("batch")

    forcing_only = set(FORCING_ONLY_VARS)
    target_vars  = [v for v in task_config.target_variables if v not in forcing_only]
    base = xr.zeros_like(ds[target_vars].isel(time=slice(-1, None)))
    targets_template = xr.concat(
        [base.assign_coords(time=[t]) for t in target_times], dim="time"
    ).expand_dims("batch")

    forcings = make_forcings(ds, target_times)
    return inputs, targets_template, forcings

inputs, targets_template, forcings = prepare_inputs_and_targets(
    era5_ds, task_config, INIT_DT, N_STEPS
)
print(f"inputs  : lat={inputs.sizes['lat']}, lon={inputs.sizes['lon']}, "
      f"level={inputs.sizes.get('level','N/A')}")
print(f"targets : {dict(targets_template.sizes)}")
print(f"forcings: {dict(forcings.sizes)}")

print(f"\nStarting inference: {INIT_DT} UTC -> +{N_STEPS*6}h ({N_STEPS} steps)")
precip_raw, _ = jit_fn(inputs, targets_template, forcings)

# Remove batch dimension
if "batch" in precip_raw.dims:
    precip_raw = precip_raw.isel(batch=0, drop=True)
print("Inference complete")
print(precip_raw)


## 🧹 Step 13. Post-processing — Unit Conversion and Cumulative Precipitation

### Unit Conversion
GraphCast outputs precipitation in **m / 6h** (meters per 6 hours).  
We multiply by 1000 to convert to the more familiar **mm / 6h**.

### Cumulative Precipitation
We sum the 6-hourly values over time to get the **accumulated precipitation**,  
which tells us the total rainfall over the entire forecast period:

```python
cumulative[t] = precip[t1] + precip[t2] + ... + precip[t]
```

This is especially useful for assessing total rainfall from typhoons.


In [ ]:
def postprocess(ds):
    result = ds.copy()
    for var in result.data_vars:
        result[var] = result[var].clip(min=0)
        if float(result[var].max()) < 1.0:
            result[var] = result[var] * 1000
            result[var].attrs["units"] = "mm"
            print(f"  {var}: m -> mm conversion")

    pv = list(result.data_vars)[0]
    result["cumulative_precipitation"] = result[pv].cumsum(dim="time")
    result["cumulative_precipitation"].attrs = {"long_name": "Cumulative Precipitation", "units": "mm"}
    print(f"  Max 6h precipitation  : {float(result[pv].max()):.2f} mm")
    print(f"  Max cumulative precip : {float(result['cumulative_precipitation'].max()):.2f} mm")
    return result

precip_vars = [v for v in precip_raw.data_vars
               if "precipitation" in v.lower() or "precip" in v.lower()]
if not precip_vars:
    precip_vars = list(precip_raw.data_vars)

precip_final = postprocess(precip_raw[precip_vars])
PRECIP_VAR   = [v for v in precip_final.data_vars if v != "cumulative_precipitation"][0]
print("\nPost-processing complete")


## 💾 Step 14. Save Results as NetCDF

We save the forecast output as **NetCDF** files.

### What is NetCDF?
NetCDF (Network Common Data Form) is the standard file format in meteorology, oceanography, and climate science.  
It stores gridded data along with coordinates (latitude, longitude, time) and metadata (units, descriptions).  
Files use the `.nc` extension and can be read by xarray, MATLAB, Python, NCO, and many GIS tools.

Two files will be saved:
- `*_global_*.nc` — full global forecast
- `*_philippines_*.nc` — Philippines region subset


In [ ]:
import os

def save_results(ds, output_dir, tag):
    os.makedirs(output_dir, exist_ok=True)
    gpath = f"{output_dir}/graphcast1deg_precip_global_{tag}.nc"
    ds.to_netcdf(gpath)
    print(f"  Global saved      : {gpath}")

    # Philippines region (lat: 4–22°N, lon: 116–128°E)
    philippines = ds.sel(lat=slice(22, 4), lon=slice(116, 128))
    ppath = f"{output_dir}/graphcast1deg_precip_philippines_{tag}.nc"
    philippines.to_netcdf(ppath)
    print(f"  Philippines saved : {ppath}")
    return gpath, ppath

global_path, philippines_path = save_results(precip_final, OUTPUT_DIR, TAG)
print("\nNetCDF save complete")


## 🗺️ Step 15. Visualization — Philippines Precipitation Forecast Map

We plot the GraphCast precipitation forecast on a map of the Philippines  
for several forecast lead times (+6h, +24h, +48h, +66h).

**How to read the map:**
- Darker colors (blue → green) indicate heavier precipitation
- Observe how the rainfall pattern evolves over time
- Notice how terrain (mountain ranges) influences precipitation distribution


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Philippines: lat 4–22°N, lon 116–128°E
phil_ds   = precip_final.sel(lat=slice(22, 4), lon=slice(116, 128))
n_sample   = min(4, N_STEPS)
step_gap   = max(1, N_STEPS // n_sample)
sample_idx = list(range(0, N_STEPS, step_gap))[:n_sample]

fig, axes = plt.subplots(
    1, len(sample_idx), figsize=(5 * len(sample_idx), 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
if len(sample_idx) == 1:
    axes = [axes]

cmap = plt.get_cmap("YlGnBu")
norm = mcolors.BoundaryNorm([0, 0.5, 1, 2, 5, 10, 20, 40, 80], cmap.N)

for ax, si in zip(axes, sample_idx):
    step_ds = phil_ds.isel(time=si)
    ax.set_extent([116, 128, 4, 22], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.6)
    ax.add_feature(cfeature.LAND,      facecolor="#f5f5f0", alpha=0.5)
    ax.add_feature(cfeature.OCEAN,     facecolor="#d6eaf8", alpha=0.5)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    gl.top_labels = gl.right_labels = False

    im = ax.pcolormesh(
        step_ds.lon, step_ds.lat, step_ds[PRECIP_VAR].values,
        cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), shading="auto",
    )
    lead_h = (si + 1) * 6
    vtime  = str(precip_final.time.values[si])[:13].replace("T", " ")
    ax.set_title(f"+{lead_h}h\n{vtime}UTC", fontsize=9)

plt.colorbar(im, ax=axes, label="Precipitation (mm/6h)", shrink=0.75)
fig.suptitle(
    f"GraphCast 1° Precipitation Forecast  |  Initial: {INIT_DT.strftime('%Y-%m-%d %H')} UTC\n"
    f"Philippines  |  2026-04-08 ~ 2026-04-10",
    fontsize=12, y=1.02,
)
plt.tight_layout()
map_path = f"{OUTPUT_DIR}/graphcast1deg_philippines_map_{TAG}.png"
plt.savefig(map_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {map_path}")


## 📈 Step 16. Visualization — Philippines Mean Precipitation Time Series

We plot the area-averaged precipitation over the Philippines as a time series.

**How to read the chart:**
- **Top (bar chart)**: 6-hourly precipitation — identifies peak rainfall periods
- **Bottom (line chart)**: accumulated precipitation — shows the total rainfall trend


In [ ]:
phil_mean = phil_ds[PRECIP_VAR].mean(dim=["lat","lon"]).values
phil_cum  = phil_ds["cumulative_precipitation"].mean(dim=["lat","lon"]).values
times_str  = [str(t)[:13].replace("T","\n") for t in precip_final.time.values]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle(
    f"GraphCast 1°  |  Philippines Mean Precipitation  |  Initial: {INIT_DT.strftime('%Y-%m-%d %H')} UTC",
    fontsize=13,
)
ax1.bar(range(N_STEPS), phil_mean, color="#2196F3", alpha=0.85, label="6h Precipitation")
ax1.set_ylabel("Precipitation (mm/6h)"); ax1.legend(); ax1.grid(axis="y", alpha=0.4)

ax2.plot(range(N_STEPS), phil_cum, "o-", color="#E91E63", linewidth=2, markersize=5, label="Accumulated Precipitation")
ax2.fill_between(range(N_STEPS), phil_cum, alpha=0.15, color="#E91E63")
ax2.set_ylabel("Accumulated Precipitation (mm)")
ax2.set_xticks(range(N_STEPS)); ax2.set_xticklabels(times_str, fontsize=7)
ax2.legend(); ax2.grid(axis="y", alpha=0.4)

plt.tight_layout()
ts_path = f"{OUTPUT_DIR}/graphcast1deg_philippines_timeseries_{TAG}.png"
plt.savefig(ts_path, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {ts_path}")


## 📊 Step 17. Forecast Results Summary Table

A table summarizing all 11 forecast steps.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Forecast Time (UTC)":              [str(t)[:16] for t in precip_final.time.values],
    "Lead Time":                        [f"+{(i+1)*6}h" for i in range(N_STEPS)],
    "Philippines 6h Precipitation (mm)":   phil_mean.round(3),
    "Philippines Cumulative Precip (mm)":  phil_cum.round(3),
})
df.index = df.index + 1; df.index.name = "Step"
print("=" * 66)
print(f"  GraphCast 1° Precipitation Forecast  |  Initial: {INIT_DT} UTC")
print(f"  Region: Philippines")
print("=" * 66)
display(df)
print(f"\n  Max 6h precipitation  : {phil_mean.max():.3f} mm")
print(f"  Total accumulated     : {phil_cum[-1]:.3f} mm")
print("=" * 66)


## ⬇️ Step 18. Download Result Files

Download the NetCDF data files and PNG figures to your local computer.


In [ ]:
from google.colab import files
import glob, os

all_files = glob.glob(f"{OUTPUT_DIR}/*.nc") + glob.glob(f"{OUTPUT_DIR}/*.png")
print("Files to download:")
for f in all_files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")
print("\nStarting download...")
for f in all_files:
    files.download(f)


---

## 🔍 Step 19. Validation — AI Forecast vs. ERA5 Reanalysis

### Why Validate?
Any forecasting model must be evaluated against observed (or best-available) data  
to understand how well it performs.  
Because we used a **past date**, the ERA5 reanalysis for the same period already exists —  
we can compare our AI forecast directly against it.

```
GraphCast forecast  (initialized 2021-08-05) ──→  predicted precipitation
                                                          ↕  compare
ERA5 reanalysis     (2021-08-05 to 2021-08-08) ──→  "actual" precipitation
```

> ⚠️ Note: ERA5 is itself a model-based reanalysis, not pure observations.  
> But it is the standard reference used to evaluate AI weather models globally.

### What to Look For in the Comparison Plots
1. Do the spatial patterns match? (does rain fall in the same areas?)
2. Are the intensities similar? (how much rain?)
3. How large is the forecast error? (GraphCast − ERA5)
4. Does the error grow with lead time? (a fundamental property of weather forecasting)


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import timedelta

# ── 1. Load ERA5 verification data for the forecast period ────────────────────
print("Loading ERA5 verification data from WeatherBench2...")
print(f"  Target times: {INIT_DT + timedelta(hours=6)} UTC  to  {INIT_DT + timedelta(hours=N_STEPS*6)} UTC")

ds_wb2 = xr.open_zarr(WB2_ZARR, storage_options={"token": "anon"}, consolidated=True)

# Build list of forecast valid times
target_times = [np.datetime64(INIT_DT + timedelta(hours=(i+1)*6)) for i in range(N_STEPS)]

# Find precipitation variable name in WB2
precip_wb2_name = None
for candidate in ["total_precipitation_6hr", "total_precipitation"]:
    if candidate in ds_wb2:
        precip_wb2_name = candidate
        break

if precip_wb2_name is None:
    raise KeyError("Precipitation variable not found in WB2 dataset.")

print(f"  WB2 precipitation variable: {precip_wb2_name}")
era5_precip_raw = ds_wb2[precip_wb2_name].sel(time=target_times).compute()

# Standardise latitude/longitude dimension names
rename = {}
if "latitude" in era5_precip_raw.dims:  rename["latitude"] = "lat"
if "longitude" in era5_precip_raw.dims: rename["longitude"] = "lon"
if rename:
    era5_precip_raw = era5_precip_raw.rename(rename)

# Regrid 0.25° → 1° by block averaging
era5_1deg = era5_precip_raw.coarsen(lat=4, lon=4, boundary="trim").mean()

# Convert m → mm if needed
if float(era5_1deg.max()) < 1.0:
    era5_1deg = era5_1deg * 1000.0

# Subset to Philippines region (4–22°N, 116–128°E)
era5_phil = era5_1deg.sel(lat=slice(22, 4), lon=slice(116, 128))

print(f"  Done: shape = {era5_phil.shape}")
print(f"  ERA5 max 6h precipitation: {float(era5_phil.max()):.2f} mm")

# ── 2. Align GraphCast forecast to the same grid ─────────────────────────────
gc_phil = phil_ds[PRECIP_VAR]  # already in mm

# Interpolate GraphCast to ERA5 grid (grids may differ slightly)
era5_lat = era5_phil.lat.values
era5_lon = era5_phil.lon.values
gc_interp = gc_phil.interp(lat=era5_lat, lon=era5_lon, method="linear")

# ── 3. Select 4 representative time steps ────────────────────────────────────
n_show   = 4
step_gap = max(1, N_STEPS // n_show)
show_idx = list(range(0, N_STEPS, step_gap))[:n_show]

# ── 4. Spatial comparison maps (GraphCast | ERA5 | Bias) ─────────────────────
print("\nDrawing spatial comparison maps...")

cmap_precip = plt.get_cmap("YlGnBu")
norm_precip = mcolors.BoundaryNorm([0, 0.5, 1, 2, 5, 10, 20, 40, 80], cmap_precip.N)
cmap_bias   = plt.get_cmap("RdBu_r")
norm_bias   = mcolors.TwoSlopeNorm(vmin=-20, vcenter=0, vmax=20)

proj = ccrs.PlateCarree()

# Extra right margin for colorbars
fig, axes = plt.subplots(
    n_show, 3,
    figsize=(16, 4.5 * n_show),
    subplot_kw={"projection": proj},
    gridspec_kw={"wspace": 0.35, "hspace": 0.25},
)
fig.suptitle(
    f"GraphCast Forecast vs ERA5 Reanalysis\n"
    f"Initial time: {INIT_DT.strftime('%Y-%m-%d %H')} UTC  |  Philippines (1° resolution)",
    fontsize=14, y=1.01,
)

def add_map_features(ax):
    ax.set_extent([116, 128, 4, 22], crs=proj)
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="#f5f5f0", alpha=0.4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    gl.top_labels = gl.right_labels = False

col_titles = ["GraphCast Forecast (mm/6h)", "ERA5 Reanalysis (mm/6h)", "Bias: GraphCast − ERA5 (mm)"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, pad=8)

im_p, im_b = None, None
for row, si in enumerate(show_idx):
    lead_h    = (si + 1) * 6
    vtime     = str(precip_final.time.values[si])[:13].replace("T", " ")
    row_label = f"+{lead_h}h\n{vtime} UTC"

    gc_step   = gc_interp.isel(time=si).values
    era5_step = era5_phil.isel(time=si).values
    bias_step = gc_step - era5_step

    for col, (data, cmap, norm) in enumerate([
        (gc_step,   cmap_precip, norm_precip),
        (era5_step, cmap_precip, norm_precip),
        (bias_step, cmap_bias,   norm_bias),
    ]):
        ax = axes[row, col]
        add_map_features(ax)
        im = ax.pcolormesh(era5_lon, era5_lat, data,
                           cmap=cmap, norm=norm,
                           transform=proj, shading="auto")
        if col < 2:
            im_p = im
        else:
            im_b = im

    # Row label on the left of each row
    axes[row, 0].text(
        -0.22, 0.5, row_label,
        transform=axes[row, 0].transAxes,
        fontsize=9, va="center", ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#e8f4fd"),
    )

# ── Colorbars: one below the left two columns, one below the right column ────
# Precipitation colorbar — spans columns 0 and 1
cbar_ax_p = fig.add_axes([0.10, -0.02, 0.52, 0.018])   # [left, bottom, width, height]
fig.colorbar(im_p, cax=cbar_ax_p, orientation="horizontal",
             label="Precipitation (mm/6h)")

# Bias colorbar — spans column 2
cbar_ax_b = fig.add_axes([0.68, -0.02, 0.24, 0.018])
fig.colorbar(im_b, cax=cbar_ax_b, orientation="horizontal",
             label="Bias (mm)")

map_cmp_path = f"{OUTPUT_DIR}/comparison_map_{TAG}.png"
plt.savefig(map_cmp_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {map_cmp_path}")

# ── 5. Time-series comparison (area mean) ────────────────────────────────────
print("\nDrawing time series comparison...")

gc_mean        = gc_interp.mean(dim=["lat", "lon"]).values
era5_mean      = era5_phil.mean(dim=["lat", "lon"]).values
steps          = np.arange(N_STEPS)
times_str      = [f"+{(i+1)*6}h" for i in range(N_STEPS)]
rmse_per_step  = np.sqrt((gc_mean - era5_mean) ** 2)
mae_per_step   = np.abs(gc_mean - era5_mean)

fig2, (ax_bar, ax_err) = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                                       gridspec_kw={"hspace": 0.35})
fig2.suptitle(
    f"GraphCast vs ERA5 — Philippines Mean Precipitation\n"
    f"Initial time: {INIT_DT.strftime('%Y-%m-%d %H')} UTC",
    fontsize=13,
)

# Top panel: side-by-side bar chart
w = 0.35
ax_bar.bar(steps - w/2, gc_mean,   w, label="GraphCast Forecast", color="#2196F3", alpha=0.85)
ax_bar.bar(steps + w/2, era5_mean, w, label="ERA5 Reanalysis",    color="#FF7043", alpha=0.85)
ax_bar.set_ylabel("6h Precipitation (mm)")
ax_bar.set_title("6-hourly Precipitation Comparison")
ax_bar.legend(loc="upper right")
ax_bar.grid(axis="y", alpha=0.4)

# Bottom panel: RMSE and MAE vs lead time
ax_err.plot(steps, rmse_per_step, "s-",  color="#9C27B0", linewidth=2,
            markersize=6, label="RMSE (mm)")
ax_err.plot(steps, mae_per_step,  "^--", color="#FF5722", linewidth=1.5,
            markersize=5, label="MAE (mm)", alpha=0.8)
ax_err.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax_err.fill_between(steps, rmse_per_step, alpha=0.12, color="#9C27B0")
ax_err.set_ylabel("Error (mm)")
ax_err.set_xlabel("Forecast Lead Time")
ax_err.set_xticks(steps)
ax_err.set_xticklabels(times_str, fontsize=8)
ax_err.set_title("Forecast Error vs Lead Time (RMSE / MAE)")
ax_err.legend(loc="upper left")
ax_err.grid(alpha=0.4)

ts_cmp_path = f"{OUTPUT_DIR}/comparison_timeseries_{TAG}.png"
plt.savefig(ts_cmp_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ts_cmp_path}")

# ── 6. Summary statistics ─────────────────────────────────────────────────────
overall_rmse = float(np.sqrt(np.mean((gc_mean - era5_mean) ** 2)))
overall_mae  = float(np.mean(np.abs(gc_mean - era5_mean)))
corr         = float(np.corrcoef(gc_mean, era5_mean)[0, 1])

print("\n" + "=" * 60)
print("  📊 GraphCast vs ERA5 Validation Summary")
print("=" * 60)
print(f"  Region  : Philippines (4–22°N, 116–128°E)")
print(f"  Steps   : {N_STEPS} × 6h")
print(f"  RMSE        : {overall_rmse:.3f} mm/6h")
print(f"  MAE         : {overall_mae:.3f} mm/6h")
print(f"  Correlation : {corr:.3f}  (closer to 1 is better)")
print()
print("  Per-step breakdown:")
import pandas as pd
df_err = pd.DataFrame({
    "Lead Time":      times_str,
    "GraphCast (mm)": gc_mean.round(3),
    "ERA5 (mm)":      era5_mean.round(3),
    "MAE (mm)":       mae_per_step.round(3),
    "RMSE (mm)":      rmse_per_step.round(3),
})
df_err.index += 1
display(df_err)
print("=" * 60)
print("\n💡 Discussion Questions:")
print("  1. Does RMSE / MAE grow consistently with lead time?")
print("  2. Is the error larger during periods of heavy rainfall?")
print("  3. A high correlation means spatial patterns match well,")
print("     but the intensities may still differ — check the maps!")
